In [1]:
pip install xgboost scikit-learn pandas numpy matplotlib seaborn shap mlflow pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.2 MB/s eta 0:00:00


In [4]:
mkdir src tests data checkpoints logs notebooks

In [5]:
import pandas as pd
import numpy as np
import os

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

def load_data(path=None):
    if path and os.path.exists(path):
        df = pd.read_csv(path)
    else:
        print("Downloading dataset from IBM GitHub...")
        df = pd.read_csv(DATA_URL)
        df.to_csv("data/raw_churn.csv", index=False)
        print(f"Saved to data/raw_churn.csv | Shape: {df.shape}")
    return df

def clean_data(df):
    df = df.copy()

    # Drop customerID - not a feature
    df.drop(columns=["customerID"], inplace=True, errors="ignore")

    # Fix TotalCharges - has spaces, convert to float
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Impute TotalCharges nulls with median (data cleaning step for the form)
    median_val = df["TotalCharges"].median()
    null_count = df["TotalCharges"].isnull().sum()
    df["TotalCharges"].fillna(median_val, inplace=True)
    print(f"Imputed {null_count} null values in TotalCharges with median={median_val:.2f}")

    # Encode binary target
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    # Encode binary Yes/No columns
    binary_cols = [
        "Partner", "Dependents", "PhoneService", "PaperlessBilling",
        "MultipleLines", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    for col in binary_cols:
        df[col] = df[col].map({"Yes": 1, "No": 0, "No phone service": 0, "No internet service": 0})

    # One-hot encode remaining categoricals
    cat_cols = ["gender", "InternetService", "Contract", "PaymentMethod"]
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    return df

def split_data(df, seed=42):
    from sklearn.model_selection import train_test_split

    X = df.drop(columns=["Churn"])
    y = df["Churn"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=seed
    )

    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"Churn rate - Train: {y_train.mean():.2%}, Val: {y_val.mean():.2%}, Test: {y_test.mean():.2%}")

    return X_train, X_val, X_test, y_train, y_val, y_test

def impute_age(df):
    """Used in unit tests - idempotent imputation."""
    df = df.copy()
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    median_val = df["TotalCharges"].median()
    df["TotalCharges"].fillna(median_val, inplace=True)
    return df

if __name__ == "__main__":
    df_raw = load_data()
    df_clean = clean_data(df_raw)
    print(df_clean.head())
    print(f"Final shape: {df_clean.shape}")

Saved to data/raw_churn.csv | Shape: (7043, 21)
Imputed 11 null values in TotalCharges with median=1397.47
   SeniorCitizen  Partner  Dependents  tenure  PhoneService  MultipleLines  \
0              0        1           0       1             0              0   
1              0        0           0      34             1              0   
2              0        0           0       2             1              0   
3              0        0           0      45             0              0   
4              0        0           0       2             1              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  ...  \
0               0             1                 0            0  ...   
1               1             0                 1            0  ...   
2               1             1                 0            0  ...   
3               1             0                 1            1  ...   
4               0             0                 0            0  ...  

/tmp/ipykernel_291/1337720602.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(median_val, inplace=True)


In [11]:
import sys
import argparse
import mlflow
import mlflow.xgboost

os.environ["PYTHONHASHSEED"] = "42"
np.random.seed(42)

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--lr",           type=float, default=0.05)
    parser.add_argument("--n_estimators", type=int,   default=300)
    parser.add_argument("--max_depth",    type=int,   default=5)
    parser.add_argument("--seed",         type=int,   default=42)
    # In a notebook environment, argparse might receive kernel-specific arguments.
    # We pass an empty list to parse_args to ignore them and use defaults.
    return parser.parse_args([])

def train():
    args = parse_args()

    # Load and prepare data
    df_raw   = load_data("data/raw_churn.csv")
    df_clean = clean_data(df_raw)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean, seed=args.seed)

    # Hyperparameter grid
    param_grid = {
        "n_estimators":  [100, 300, 500],
        "max_depth":     [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample":     [0.7, 0.9],
    }

    # Stratified CV - critical because churn is imbalanced (~26%)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=args.seed)
    clf = XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=args.seed
    )

    print("Starting GridSearchCV (this takes ~5-10 minutes)...")
    grid_search = GridSearchCV(
        clf, param_grid, cv=cv,
        scoring="roc_auc", n_jobs=4, verbose=1
    )
    grid_search.fit(X_train, y_train)

    best_model  = grid_search.best_estimator_
    best_params = grid_search.best_params_
    cv_auc      = grid_search.best_score_

    # Evaluate on validation set
    val_preds = best_model.predict_proba(X_val)[:, 1]
    val_auc   = roc_auc_score(y_val, val_preds)
    val_preds_binary = (val_preds >= 0.4).astype(int)

    # Train AUC (to check overfitting)
    train_preds = best_model.predict_proba(X_train)[:, 1]
    train_auc   = roc_auc_score(y_train, train_preds)

    print(f"\n[RESULTS] CV AUC={cv_auc:.4f} | Train AUC={train_auc:.4f} | Val AUC={val_auc:.4f}")
    print(f"Best params: {best_params}")
    print(f"\nClassification Report (threshold=0.4):")
    print(classification_report(y_val, val_preds_binary, target_names=["Retain", "Churn"]))

    # Save checkpoint
    os.makedirs("checkpoints", exist_ok=True)
    checkpoint_path = "checkpoints/xgb_best_v1.json"
    best_model.save_model(checkpoint_path)
    print(f"\nModel saved: {checkpoint_path}")

    # Log to MLflow
    mlflow.set_experiment("churn_prediction")
    with mlflow.start_run() as run:
        mlflow.log_params(best_params)
        mlflow.log_metric("cv_auc",    cv_auc)
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc",   val_auc)
        mlflow.xgboost.log_model(best_model, "model")
        run_id = run.info.run_id
        print(f"\nMLflow Run ID: {run_id}")

    # Save log line for the form
    os.makedirs("logs", exist_ok=True)
    with open("logs/train_results.log", "w") as f:
        f.write(f"[FINAL] val_auc={val_auc:.4f} | train_auc={train_auc:.4f} | "
                f"best_params={best_params} | checkpoint=xgb_best_v1.json | "
                f"mlflow_run_id={run_id}\n")

    print("\nDone. Check logs/train_results.log for final validation line.")

train()

Imputed 11 null values in TotalCharges with median=1397.47
Train: (4932, 23), Val: (1054, 23), Test: (1057, 23)
Churn rate - Train: 26.54%, Val: 26.57%, Test: 26.49%
Starting GridSearchCV (this takes ~5-10 minutes)...
Fitting 5 folds for each of 54 candidates, totalling 270 fits


/tmp/ipykernel_291/1337720602.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(median_val, inplace=True)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:20:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[RESULTS] CV AUC=0.8507 | Train AUC=0.8676 | Val AUC=0.8363
Best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.7}

Classification Report (threshold=0.4):
              precision    recall  f1-score   support

      Retain       0.86      0.86      0.86       774
       Churn       0.61      0.61      0.61       280

    accuracy                           0.79      1054
   macro avg       0.73      0.73      0.73      1054
weighted avg       0.79      0.79      0.79      1054


Model saved: checkpoints/xgb_best_v1.json


2026/03/15 19:20:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/15 19:20:38 INFO mlflow.store.db.utils: Updating database tables
2026/03/15 19:20:40 INFO mlflow.tracking.fluent: Experiment with name 'churn_prediction' does not exist. Creating a new experiment.
2026/03/15 19:20:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



MLflow Run ID: 77196fb25bc244bfa9adfe4012cf15ab

Done. Check logs/train_results.log for final validation line.


In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)


def evaluate():
    df_raw   = load_data("data/raw_churn.csv")
    df_clean = clean_data(df_raw)
    _, X_val, X_test, _, y_val, y_test = split_data(df_clean)

    # Load saved model
    model = XGBClassifier()
    model.load_model("checkpoints/xgb_best_v1.json")

    # Validation metrics
    val_probs  = model.predict_proba(X_val)[:, 1]
    val_preds  = (val_probs >= 0.4).astype(int)
    val_auc    = roc_auc_score(y_val, val_probs)

    print(f"Validation AUC: {val_auc:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_val, val_preds)
    print(f"\nConfusion Matrix:\n{cm}")
    print(f"TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

    # Error analysis - find the failure mode
    import pandas as pd
    val_df = X_val.copy()
    val_df["true_label"]  = y_val.values
    val_df["pred_label"]  = val_preds
    val_df["pred_proba"]  = val_probs

    # False negatives - churners we missed
    false_negatives = val_df[(val_df["true_label"]==1) & (val_df["pred_label"]==0)]
    print(f"\nFalse Negatives (missed churners): {len(false_negatives)}")
    print("Mean feature values for missed churners:")
    print(false_negatives[["tenure", "MonthlyCharges", "TotalCharges"]].mean())

    # Save confusion matrix plot
    os.makedirs("logs", exist_ok=True)
    fig, ax = plt.subplots(figsize=(6,5))
    disp = ConfusionMatrixDisplay(cm, display_labels=["Retain","Churn"])
    disp.plot(ax=ax, colorbar=False)
    plt.title("Confusion Matrix (threshold=0.4)")
    plt.tight_layout()
    plt.savefig("logs/confusion_matrix.png", dpi=150)
    print("Saved: logs/confusion_matrix.png")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_val, val_probs)
    fig2, ax2 = plt.subplots(figsize=(6,5))
    ax2.plot(fpr, tpr, label=f"AUC = {val_auc:.4f}")
    ax2.plot([0,1],[0,1],"--", color="gray")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve - Validation Set")
    ax2.legend()
    plt.tight_layout()
    plt.savefig("logs/roc_curve.png", dpi=150)
    print("Saved: logs/roc_curve.png")


evaluate()

/tmp/ipykernel_291/1337720602.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(median_val, inplace=True)


Imputed 11 null values in TotalCharges with median=1397.47
Train: (4932, 23), Val: (1054, 23), Test: (1057, 23)
Churn rate - Train: 26.54%, Val: 26.57%, Test: 26.49%
Validation AUC: 0.8363

Confusion Matrix:
[[665 109]
 [109 171]]
TN=665, FP=109, FN=109, TP=171

False Negatives (missed churners): 109
Mean feature values for missed churners:
tenure              33.036697
MonthlyCharges      66.723394
TotalCharges      2690.134862
dtype: float64
Saved: logs/confusion_matrix.png
Saved: logs/roc_curve.png
